<a href="https://colab.research.google.com/github/ariannawilde3/CPE315_LAB2/blob/main/Copy_of_%5BCODE_TEMPLATE%5D_ESE_5420_Homework9_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ESE 5420 - Homework 9
**Author:** `Arianna Wilde`

Please make a LOCAL copy of this otherwise you can't save.

In [ ]:
# These are the only imports you can use for Question 1 and Question 2
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(5420)

## Question 1: Logistic Regression from Scratch

Load the data

In [ ]:
# TO-DO: Use np.load(...) to read the data then assign the data to X, and the labels to y.
X = np.load("data.npy")
y = np.load("label.npy").reshape(-1)

X = X.reshape(len(y), 784)

print("Data shape:", X.shape)
print("Labels shape:", y.shape)

Visualize one input data point as an image for each of label 0 and label 1.

In [ ]:
# TO-DO: The data should be reshaped back to [28 x 28] to be able to visualize it using plt.imshow()
fig, axes = plt.subplots(1, 2, figsize=(6, 3))

for label in [0, 1]:
    index = np.flatnonzero(y == label)[0]
    axes[label].imshow(
        X[index].reshape(28, 28),
        cmap="gray",
        vmin=0,
        vmax=255
    )
    axes[label].set_title(f"Original label: {label}")
    axes[label].axis("off")

plt.tight_layout()
plt.show()

Since the data is in between 0 to 255, normalize the data to [0, 1]

In [ ]:
# TO-DO: Normalize the data
X = X.astype(np.float64) / 255.0

Set $y_i = +1$ for images originally labeled 0, and $y_i = -1$ for images originally labeled 1.

In [ ]:
# TO-DO: Convert labels
y = np.where(y == 0, 1, -1)

Split the data randomly into train and test with a ratio of 80:20.

In [ ]:
# TO-DO: Create function that takes in (X,y) and splits the data by returning (X_train, y_train, X_test, y_test)
def split_data(X, y):
    indices = np.random.permutation(len(y))
    split = int(0.8 * len(y))

    train_indices = indices[:split]
    test_indices = indices[split:]

    return (
        X[train_indices],
        y[train_indices],
        X[test_indices],
        y[test_indices]
    )

In [ ]:
# TO-DO: Call your function to perform the Train-Test Split on our data
X_train, y_train, X_test, y_test = split_data(X, y)

print("Training samples:", len(y_train))
print("Testing samples:", len(y_test))

Initialize the coefficients $\beta_0^{(1)}$, **$\vec{\beta}$** using a Normal Distribution of mean 0 and variance 1.

For **$\beta_1$**, initialize all $d$ entries to be $N(0,1)$)

In [ ]:
# TO-DO: Initialize all d entries to be sampled from a standard normal distribution
B = np.random.randn(X.shape[1])
B_0 = float(np.random.randn())

Compute the loss function
$$L(\beta_0^{(1)}, \vec{\beta}) = \frac{1}{m}\sum_{i=1}^m \text{ln}(1+e^{-y_i (\beta_0 + \sum_{j=1}^d \beta_{j}x_{i,j})})$$
where $x_{i,j}$ is the $j$-th of entry data point $\vec{\textbf{x}_i}$

In [ ]:
# TO DO: Helper function to compute loss
def compute_loss(data, labels, B, B_0):
    scores = data @ B + B_0
    margins = labels * scores

    # Stable computation of log(1 + exp(-margin)).
    return np.mean(np.logaddexp(0.0, -margins))

Compute the gradients of the loss function

$$\frac{\partial L}{\partial \beta_0} = d\beta_0 = -\frac{1}{m}\sum_{i=1}^m \frac{e^{-y_i \cdot (\beta_0 + \vec{\beta^T} \vec{x_{i}})}}{1 + e^{-y_i (\beta_0 + \vec{\beta^T} \vec{x_{i}})}} y_i$$
$$\nabla_\beta L = d\vec{\beta} = -\frac{1}{m}\sum_{i=1}^m \frac{e^{-y_i \cdot (\beta_0 + \vec{\beta^T} \vec{x_{i}})}}{1 + e^{-y_i (\beta_0 + \vec{\beta^T} \vec{x_{i}})}} y_i x_i$$

In [ ]:
# TO-DO: Helper function to compute loss
def compute_gradients(data, labels, B, B_0):
    scores = data @ B + B_0
    margins = labels * scores

    # Stable computation of 1 / (1 + exp(margin)).
    weights = np.exp(-np.logaddexp(0.0, margins))
    residuals = -labels * weights

    dB = data.T @ residuals / len(labels)
    dB_0 = np.mean(residuals)

    return dB, dB_0


Update the parameters using gradient updates from the train set as:
$$\beta_j \leftarrow \beta_j - 0.05 \cdot d\beta_j$$
$$\beta_0 \leftarrow \beta_0 - 0.05 \cdot d\beta_0$$

Repeat the process for 50 iterations. You should save your results for each of the 50 epochs in `accuracy_hist`, `train_loss_hist`, and `test_loss_hist`.

In [ ]:
# TO-DO: Write your code below. This will be the longest part!
lr = 0.05
num_epochs = 50

accuracy_hist = []
train_loss_hist = []
test_loss_hist = []

for epoch in range(num_epochs):
    # Compute both gradients using the current parameters.
    dB, dB_0 = compute_gradients(X_train, y_train, B, B_0)

    # Update parameters using only training data.
    B = B - lr * dB
    B_0 = B_0 - lr * dB_0

    # Evaluate after this epoch's update.
    train_loss = compute_loss(X_train, y_train, B, B_0)
    test_loss = compute_loss(X_test, y_test, B, B_0)

    test_scores = X_test @ B + B_0
    test_predictions = np.where(test_scores >= 0.0, 1, -1)
    test_accuracy = np.mean(test_predictions == y_test)

    train_loss_hist.append(train_loss)
    test_loss_hist.append(test_loss)
    accuracy_hist.append(test_accuracy)

print(f"Training loss after 50 epochs: {train_loss_hist[-1]:.6f}")
print(f"Testing loss after 50 epochs: {test_loss_hist[-1]:.6f}")
print(f"Testing accuracy after 50 epochs: {accuracy_hist[-1]:.2%}")

Plot your training and testing loss curves side-by-side below by running the plotting code given below. You don't need to modify these two cells.

In [ ]:
fig = plt.figure(figsize = (15,6))

# Plot Loss Function
ax1 = fig.add_subplot(121)
ax1.plot(train_loss_hist);
ax1.set_title("Training Loss over each Iteration", fontsize = 16);
ax1.set_xlabel("Iteration");
ax1.set_ylabel("Loss");

# Plot Accuracy Function
ax2 = fig.add_subplot(122)
ax2.plot(test_loss_hist);
ax2.set_title("Testing Loss over each Iteration", fontsize = 16);
ax2.set_xlabel("Iteration");
ax2.set_ylabel("Loss");

In [ ]:
# Plot your accuracy curve below by running the template code below
plt.plot(accuracy_hist);
plt.title("Testing Accuracy over each Iteration");
plt.xlabel("Iteration");
plt.ylabel("Accuracy");

## Congratulations!!! Now submit this in gradescope!

You're done with all the coding questions! Now submit the code/images generated to the respective questions in gradescope. Also, submit this notebook by following the steps below.

1. Download your Colab notebook as an `.ipynb` file to save locally on your computer
2. Submit the .ipynb file on gradescope


Leave enough time to ask for help if you're stuck so you don't get hit with the Late Penalty!